# AquaRakshak Calibration and Accuracy
Use this notebook to compute accuracy metrics from your dataset and present evidence to judges.

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support

In [ ]:
csv_path = 'your_dataset.csv'  # update path
df = pd.read_csv(csv_path)
df.head()

In [ ]:
# Required columns: ph, turbidity, tds, waterLevel, flowRate, label
# label in: contamination, leakage, shortage, safe
def predict(row):
    ph, turb, tds, wl, fr = row['ph'], row['turbidity'], row['tds'], row['waterLevel'], row['flowRate']
    ph_s = 0 if 6.5 <= ph <= 8.5 else min(1, (6.5-ph)/3 if ph < 6.5 else (ph-8.5)/3)
    turb_s = 0 if turb <= 5 else min(1, (turb-5)/20)
    tds_s = 0 if tds <= 500 else min(1, (tds-500)/1500)
    contamination = min(1, 0.4*ph_s + 0.35*turb_s + 0.25*tds_s)
    shortage = 0 if wl >= 20 else min(1, (20-wl)/20)
    leakage = 0 if fr >= 1 else min(1, (1-fr)/1)
    scores = {'contamination': contamination, 'shortage': shortage, 'leakage': leakage}
    top = max(scores, key=scores.get)
    return top if scores[top] >= 0.62 else 'safe'

df['predicted'] = df.apply(predict, axis=1)
df[['label', 'predicted']].head()

In [ ]:
labels = sorted(df['label'].dropna().unique())
print('Confusion Matrix')
print(confusion_matrix(df['label'], df['predicted'], labels=labels))
print('\nClassification Report')
print(classification_report(df['label'], df['predicted'], labels=labels, zero_division=0))

In [ ]:
p, r, f, s = precision_recall_fscore_support(df['label'], df['predicted'], labels=labels, zero_division=0)
kpi = pd.DataFrame({'label': labels, 'precision': p, 'recall': r, 'f1': f, 'support': s})
kpi